# Pipeline DOIs → Título/Abstract → PDFs

Este notebook hace tres cosas principales:

1. A partir de un archivo de texto desordenado, extrae todos los **DOIs** y genera `papers_info.csv`.
2. Usa la API de **Semantic Scholar** para obtener **título** y **abstract** de cada DOI y genera `papers_info_with_text.csv`.
3. Intenta descargar el **PDF** de cada artículo desde:
   - Semantic Scholar (openAccessPdf),
   - arXiv,
   - Europe PMC / PubMed Central,
   - Unpaywall (opcional, si se configura email),
   y genera el archivo final `pdf_status.csv`.

Para usar este notebook:
- Solo necesitas cambiar los nombres de archivos en la celda de **Configuración**.
- Luego ejecutar las celdas **en orden**, de arriba hacia abajo.


## Configuración

In [ ]:
from __future__ import annotations

from pathlib import Path

# -------------------------
# Configuración de archivos
# -------------------------

# Archivo de texto original con referencias desordenadas
INPUT_TEXT_FILE = Path("summary-stomachneo-set.txt")

# CSV intermedios
DOI_CSV_FILE = Path("papers_info.csv")
INFO_CSV_FILE = Path("papers_info_with_text.csv")

# Carpeta donde se guardarán los PDFs
PDF_FOLDER = Path("pdfs")

# CSV final con el estado de los PDFs
PDF_STATUS_FILE = Path("pdf_status.csv")

# -------------------------
# APIs externas
# -------------------------

from semanticscholar import SemanticScholar
import requests
import csv
import re
from typing import Optional, Dict, Tuple, List

# Si tienes API key de Semantic Scholar, colócala aquí:
# sch = SemanticScholar(api_key="TU_API_KEY")
sch = SemanticScholar()

# Unpaywall (opcional). Deja "" si no quieres usar Unpaywall.
UNPAYWALL_EMAIL = "tu_email@ejemplo.com"

# Crear carpeta de PDFs si no existe
PDF_FOLDER.mkdir(exist_ok=True)


## Funciones

In [ ]:
# -------------------------
# Utilidades generales
# -------------------------

def extract_dois_from_text(text: str) -> List[str]:
    """
    Extrae DOIs usando una expresión regular y elimina duplicados.
    """
    pattern = r"10\.\d{4,9}/[-._;()/:A-Za-z0-9]+"
    raw_dois = re.findall(pattern, text)

    # Limpiar cosas como "https://doi.org/"
    cleaned = [doi.split("doi.org/")[-1] for doi in raw_dois]

    # Eliminar duplicados manteniendo el orden
    unique_dois = list(dict.fromkeys(cleaned))
    return unique_dois


def normalize_doi(raw: Optional[str]) -> str:
    """
    Deja el DOI en un formato estándar, sin prefijos ni puntuación final.
    """
    if not raw:
        return ""
    doi = raw.strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("doi.org/", "")
    doi = doi.lstrip("(")
    doi = doi.rstrip(").,;:")
    return doi


def download_pdf(url: str, dest_path: Path) -> bool:
    """
    Descarga un PDF desde una URL y lo guarda en dest_path.
    Retorna True si tuvo éxito.
    """
    try:
        r = requests.get(url, timeout=30)
        content_type = (r.headers.get("Content-Type") or "").lower()
        if r.status_code == 200 and "pdf" in content_type:
            with open(dest_path, "wb") as f:
                f.write(r.content)
            return True
    except requests.RequestException:
        return False
    return False


## Pipeline completo

In [ ]:
# -------------------------
# PASO 1: Texto → DOIs → papers_info.csv
# -------------------------

def step1_extract_dois(
    input_text_file: Path = INPUT_TEXT_FILE,
    output_csv: Path = DOI_CSV_FILE,
) -> None:
    print(f"Leyendo texto desde: {input_text_file}")
    text = input_text_file.read_text(encoding="utf-8")

    dois = extract_dois_from_text(text)
    print(f"Se encontraron {len(dois)} DOIs.")

    with output_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["doi"])
        writer.writeheader()
        for doi in dois:
            writer.writerow({"doi": doi})

    print(f"Archivo creado: {output_csv}")


# Ejecutar Paso 1
step1_extract_dois()


Leyendo texto desde: summary-stomachneo-set.txt
Se encontraron 98 DOIs.
Archivo creado: papers_info.csv


In [ ]:
# -------------------------
# PASO 2: DOIs → título + abstract → papers_info_with_text.csv
# -------------------------

def fetch_paper_metadata(doi: str) -> Tuple[str, str]:
    """
    Devuelve (title, abstract) para un DOI.
    Si no encuentra datos, devuelve ("", "").
    """
    doi_norm = normalize_doi(doi)
    try:
        paper = sch.get_paper(doi_norm, fields=["title", "abstract"])
    except Exception as e:
        print(f"  Error al consultar Semantic Scholar para {doi_norm}: {e}")
        return "", ""

    if not paper:
        return "", ""

    title = getattr(paper, "title", "") or ""
    abstract = getattr(paper, "abstract", "") or ""
    return title, abstract


def step2_build_info_csv(
    input_csv: Path = DOI_CSV_FILE,
    output_csv: Path = INFO_CSV_FILE,
) -> None:
    print(f"Leyendo DOIs desde: {input_csv}")
    with input_csv.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        dois = [normalize_doi(row["doi"]) for row in reader if row.get("doi")]

    rows = []
    for doi in dois:
        print(f"\nBuscando metadata para DOI: {doi}")
        title, abstract = fetch_paper_metadata(doi)
        rows.append(
            {
                "doi": doi,
                "title": title,
                "abstract": abstract,
            }
        )

    with output_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["doi", "title", "abstract"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nArchivo creado: {output_csv}")


# Ejecutar Paso 2
step2_build_info_csv()


Leyendo DOIs desde: papers_info.csv

Buscando metadata para DOI: 10.1080/15548627.2021.1901204

Buscando metadata para DOI: 10.3390/cancers14205105

Buscando metadata para DOI: 10.62347/BVFO4627

Buscando metadata para DOI: 10.1016/j.clinbiochem.2024.110767

Buscando metadata para DOI: 10.1016/j.cca.2024.117773

Buscando metadata para DOI: 10.1200/PO.22.00420

Buscando metadata para DOI: 10.1007/s10120-022-01313-w

Buscando metadata para DOI: 10.3389/fonc.2024.1341056

Buscando metadata para DOI: 10.1007/s10120-024-01556-9

Buscando metadata para DOI: 10.3389/fgene.2024.1425591

Buscando metadata para DOI: 10.2174/1386207325666220616125608

Buscando metadata para DOI: 10.1159/000524283

Buscando metadata para DOI: 10.1016/j.phymed.2024.156137

Buscando metadata para DOI: 10.1159/000514457

Buscando metadata para DOI: 10.1186/s13046-024-03043-6

Buscando metadata para DOI: 10.23736/S0026-4806.20.06628-8

Buscando metadata para DOI: 10.1007/s13258-023-01412-7

Buscando metadata para DOI:

In [ ]:
# -------------------------
# PASO 2: DOIs → título + abstract → papers_info_with_text.csv
# -------------------------

def fetch_paper_metadata(doi: str) -> Tuple[str, str]:
    """
    Devuelve (title, abstract) para un DOI.
    Si no encuentra datos, devuelve ("", "").
    """
    doi_norm = normalize_doi(doi)
    try:
        paper = sch.get_paper(doi_norm, fields=["title", "abstract"])
    except Exception as e:
        print(f"  Error al consultar Semantic Scholar para {doi_norm}: {e}")
        return "", ""

    if not paper:
        return "", ""

    title = getattr(paper, "title", "") or ""
    abstract = getattr(paper, "abstract", "") or ""
    return title, abstract


def step2_build_info_csv(
    input_csv: Path = DOI_CSV_FILE,
    output_csv: Path = INFO_CSV_FILE,
) -> None:
    print(f"Leyendo DOIs desde: {input_csv}")
    with input_csv.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        dois = [normalize_doi(row["doi"]) for row in reader if row.get("doi")]

    rows = []
    for doi in dois:
        print(f"\nBuscando metadata para DOI: {doi}")
        title, abstract = fetch_paper_metadata(doi)
        rows.append(
            {
                "doi": doi,
                "title": title,
                "abstract": abstract,
            }
        )

    with output_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["doi", "title", "abstract"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nArchivo creado: {output_csv}")


# Ejecutar Paso 2
step2_build_info_csv()


In [ ]:
# -------------------------
# Funciones específicas para PDFs
# -------------------------

EUROPE_PMC_BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest"


def try_semantic_scholar_pdf(doi: str, pdf_path: Path) -> bool:
    """
    Intenta obtener PDF vía openAccessPdf de Semantic Scholar.
    """
    doi_norm = normalize_doi(doi)
    try:
        paper = sch.get_paper(doi_norm, fields=["openAccessPdf", "externalIds"])
    except Exception:
        return False

    if not paper:
        return False

    pdf_info = getattr(paper, "openAccessPdf", None)
    if pdf_info and isinstance(pdf_info, dict):
        url = pdf_info.get("url")
        if url:
            print(f"  Intentando PDF desde Semantic Scholar: {url}")
            return download_pdf(url, pdf_path)
    return False


def try_arxiv_pdf(doi: str, pdf_path: Path) -> bool:
    """
    Si el artículo tiene ID de arXiv en Semantic Scholar,
    intenta descargar el PDF de arXiv.
    """
    doi_norm = normalize_doi(doi)
    try:
        paper = sch.get_paper(doi_norm, fields=["externalIds"])
    except Exception:
        return False

    if paper is None:
        return False

    external_ids = getattr(paper, "externalIds", None)
    if not external_ids:
        return False

    arxiv_id = (
        external_ids.get("ArXiv")
        or external_ids.get("Arxiv")
        or external_ids.get("ARXIV")
    )
    if not arxiv_id:
        return False

    url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    print(f"  Intentando PDF desde arXiv: {url}")
    return download_pdf(url, pdf_path)


def fetch_europe_pmc_record(doi: str) -> Optional[dict]:
    params = {
        "query": f"DOI:{normalize_doi(doi)}",
        "resulttype": "core",
        "format": "json",
        "pageSize": 1,
    }
    try:
        resp = requests.get(f"{EUROPE_PMC_BASE}/search", params=params, timeout=30)
        if not resp.ok:
            return None
        data = resp.json()
    except requests.RequestException:
        return None

    results = data.get("resultList", {}).get("result", [])
    if not results:
        return None
    return results[0]


def get_pdf_url_from_epmc_record(record: dict) -> Optional[str]:
    urls = record.get("fullTextUrlList", {}).get("fullTextUrl", [])
    for item in urls:
        style = (item.get("documentStyle") or "").lower()
        url = item.get("url") or ""
        if style == "pdf" and url:
            return url
    return None


def try_epmc_pdf(doi: str, pdf_path: Path) -> bool:
    record = fetch_europe_pmc_record(doi)
    if not record:
        return False

    # 1) URL directa a PDF
    url = get_pdf_url_from_epmc_record(record)

    # 2) Fallback: construir URL desde PMCID
    if not url:
        pmcid = record.get("pmcid")
        if pmcid:
            url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/pdf/"

    if not url:
        return False

    print(f"  Intentando PDF desde Europe PMC / PMC: {url}")
    return download_pdf(url, pdf_path)


def try_unpaywall_pdf(doi: str, pdf_path: Path) -> bool:
    """
    Intenta obtener PDF desde Unpaywall (solo si UNPAYWALL_EMAIL está definido).
    """
    if not UNPAYWALL_EMAIL:
        return False

    url = f"https://api.unpaywall.org/v2/{normalize_doi(doi)}"
    params = {"email": UNPAYWALL_EMAIL}

    try:
        resp = requests.get(url, params=params, timeout=30)
        if not resp.ok:
            return False
        data = resp.json()
    except requests.RequestException:
        return False

    best = data.get("best_oa_location") or {}
    pdf_url = best.get("url_for_pdf") or best.get("url")
    if not pdf_url:
        return False

    print(f"  Intentando PDF desde Unpaywall: {pdf_url}")
    return download_pdf(pdf_url, pdf_path)


def download_pdf_for_doi(doi: str) -> Tuple[str, str]:
    """
    Intenta descargar el PDF para un DOI usando, en orden:
      1) Semantic Scholar
      2) arXiv
      3) Europe PMC / PMC
      4) Unpaywall

    Devuelve (pdf_local, pdf_path_str), donde:
      - pdf_local es "sí" o "no"
      - pdf_path_str es la ruta al PDF o "" si no se obtuvo.
    """
    doi_norm = normalize_doi(doi)
    pdf_path = PDF_FOLDER / f"{doi_norm.replace('/', '_')}.pdf"

    # Si ya existe, no volver a descargar
    if pdf_path.exists():
        print("  PDF ya existía localmente.")
        return "sí", str(pdf_path)

    # Orden de intentos
    if try_semantic_scholar_pdf(doi_norm, pdf_path):
        return "sí", str(pdf_path)

    if try_arxiv_pdf(doi_norm, pdf_path):
        return "sí", str(pdf_path)

    if try_epmc_pdf(doi_norm, pdf_path):
        return "sí", str(pdf_path)

    if try_unpaywall_pdf(doi_norm, pdf_path):
        return "sí", str(pdf_path)

    # Si nada funcionó
    return "no", ""


In [ ]:
# -------------------------
# PASO 3: Descargar PDFs y generar pdf_status.csv
# -------------------------

def load_title_abstract_by_doi(info_csv: Path = INFO_CSV_FILE) -> Dict[str, Tuple[str, str]]:
    """
    Carga un diccionario: doi → (title, abstract)
    desde papers_info_with_text.csv
    """
    mapping: Dict[str, Tuple[str, str]] = {}
    with info_csv.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            doi = normalize_doi(row.get("doi"))
            title = row.get("title", "")
            abstract = row.get("abstract", "")
            if doi:
                mapping[doi] = (title, abstract)
    return mapping


def step3_download_pdfs_and_build_status(
    doi_csv: Path = DOI_CSV_FILE,
    info_csv: Path = INFO_CSV_FILE,
    status_csv: Path = PDF_STATUS_FILE,
) -> None:
    titles_map = load_title_abstract_by_doi(info_csv)

    rows = []

    with doi_csv.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        dois = [normalize_doi(row["doi"]) for row in reader if row.get("doi")]

    for doi in dois:
        print(f"\nProcesando DOI: {doi}")

        title, abstract = titles_map.get(doi, ("", ""))

        pdf_local, pdf_path_str = download_pdf_for_doi(doi)

        rows.append(
            {
                "doi": doi,
                "title": title,
                "abstract": abstract,
                "pdf_local": pdf_local,
                "pdf_path": pdf_path_str,
            }
        )

    with status_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["doi", "title", "abstract", "pdf_local", "pdf_path"],
        )
        writer.writeheader()
        writer.writerows(rows)

    total = len(rows)
    with_pdf = sum(1 for r in rows if r["pdf_local"] == "sí")
    print(f"\nListo. PDFs locales: {with_pdf}/{total} (~{with_pdf/total*100:.1f}%).")
    print(f"Archivo de estado creado: {status_csv}")


# Ejecutar Paso 3
step3_download_pdfs_and_build_status()
